# NB07 — L4 + L5: hipótesis → resultados y registro numérico → paper

Capa **L4** (coverage-denominator: hipótesis → `EXP-*` → `results.vsm`) y
**L5** (registro numérico `paper_data.vsm` ↔ `main.tex` ↔ `addendum.tex`)
del plan auto-ordenado de NB03. Además cierra **L3** con el check 9
(`lipschitz_deployed` — constantes deployed en `operator.rs`, el check
que NB06 dejó pendiente de código).

**Determinismo (R10):** sin aleatoriedad, sin reloj, orden fijo.
**Locality (R1):** todos los paths resueltos walk-up desde el archivo del
helper, nunca desde cwd.

**Helpers:** `l3_verify.py` (9 checks), `hypo_join.py` (L4),
`numeric_join.py` (L5 — registry-driven + cruce addendum↔paper + layer2a
abstract↔body).

## ① Cierre de L3 — 9 checks (los 8 de NB06 + `lipschitz_deployed`)

In [ ]:
import sys, json
sys.path.insert(0, ".")
from l3_verify import run_all, summary
rows = run_all()
for r in rows:
    print(f"[{r['verdict']:12s}] {r['id']:24s} <- {r['source']}")
print("summary:", json.dumps(summary(rows)["by_verdict"]))
lf = next(r for r in rows if r["id"] == "lipschitz_deployed")
print()
print("=== lipschitz_deployed ===")
print(" claim:", lf["claim"])
for e in lf["evidence"]:
    print(" -", e)

[PASS        ] pf_block_spectrum        <- eq:pf_matrix,eq:pf_lyap
[PASS        ] lambda_j_degenerate      <- eq:lambda_max
[PASS        ] lyapunov_positive        <- eq:lyapunov
[CONDITIONAL ] convergence_conditional  <- eq:convergence
[PASS        ] landauer_identity        <- cor:lyaplandauer
[PASS        ] s5_min_semantics         <- eq:v,prop:s5_deterministic
[PASS        ] margin_frontier          <- eq:margin
[PASS_GRID   ] cone_convex_grid         <- thm:hybrid_invariance
[PASS        ] lipschitz_deployed       <- thm:hybrid_invariance
summary: {"PASS": 7, "CONDITIONAL": 1, "PASS_GRID": 1}

=== lipschitz_deployed ===
 claim: L_f ∈ [1.4, 24.4] sobre τ deployed; L_f > 1 en todas las fases (L≤1 NO descargado por constantes)
 - max‖∇logit‖ = 1.118034 vs 1.118034 -> match@6dp=True; diam = 1.092016 vs 1.092016 (mu_s1−mu_s3) -> match@6dp=True
 - parse operator.rs (5 modos, 5 logits, τ=1.05−lifecycle): τ=0.85:L_f=1.436, τ=0.55:L_f=2.220, τ=0.35:L_f=3.488, τ=0.05:L_f=24.418
 - vs REVIEW

### Lectura — check 9 (`lipschitz_deployed`)

- **Parsea** `projects/vsf-nucleus/src/operator.rs` (5 modos, 5 logits,
  `τ = 1.05 − lifecycle`) — constantes *deployed*, no del paper.
- `max‖∇logit‖ = 1.118034`, `diam = 1.092016` (μ_S1−μ_S3) — match@6dp;
  **L_f = 1.436 / 2.220 / 3.488 / 24.418** @3dp contra
  `REVIEW_2026-07-11` y rango `[1.4, 24.4]` vs addendum §2 — exacto.
- **L_f > 1 en 4/4 fases** → el item (3) de `thm:hybrid_invariance`
  (L≤1) queda **no reclamado por las constantes** — teorema honesto,
  coherente=True. El paper no declara contractividad donde no la hay.
- L3 total: **7 PASS + 1 CONDITIONAL + 1 PASS_GRID** (9 checks).

## ② L4 — hipótesis → `EXP-*` → `results.vsm` (coverage denominator)

In [ ]:
import sys, json
sys.path.insert(0, ".")
from hypo_join import run as hypo_run
rows, cov = hypo_run()
print(f"{'id':8s} {'kind':8s} {'section':26s} {'exp':13s} {'res':15s} {'v':4s} verdict")
for r in rows:
    sec = r["section"].replace("sec:", "")
    vt = (r["verdict_text"] or "-")[:24]
    print(f"{r['id']:8s} {r['kind']:8s} {sec:26s} "
          f"{str(r['exp'] or '-'):13s} {r['resolution']:15s} "
          f"{r['n_values']:4d} {vt}")
print()
print(json.dumps(cov, indent=1))

id       kind     section                    exp           res             v    verdict
H1       table    real_proc_gov              EXP-L1-01     exact             18 PASS — vsf-nucleus spawn
H2       table    real_proc_gov              EXP-L1-01     exact             18 PASS — cpu.weight is mon
H4       table    real_proc_gov              EXP-L1-01     exact             18 VIABLE (nominal threshol
H5       table    real_proc_gov              EXP-L1-01     exact             18 PASS — median cpu headro
H_linear named    thermal_ladder             -             protocol          39 REFUTED — Δcos/ΔT ≈ 0 ac
H_nonlinear named    thermal_ladder             -             protocol          39 WEAK — slight non-linear
H_floor  named    thermal_ladder             -             protocol          39 CONFIRMED — basin floor 
H1       finding  thermal_cycling            EXP-TC-01     exp_no_results     0 -
H2       finding  thermal_cycling            EXP-TC-01     exp_no_results     0 -
H3       f

### Lectura — L4

- **15 hipótesis** detectadas en `main.tex` (3 named §thermal_ladder,
  4 de tabla §real_proc_gov [H1/H2/H4/H5 — H3 no está en la tabla],
  8 finding cycling+maxwell).
- **Coverage 11/15 = 0.7333** — con archivo, veredicto Y valores: 11/11/11
  (el join no infla: archivo vacío ≠ evidencia).
- Resoluciones: **exact 8, protocol 3** (token/ambiguous/none = 0).
- **Gaps: 4 = todos `sec:thermal_cycling` → EXP-TC-01** — el directorio
  `experiments/thermal_cycling/` **no tiene `results.vsm`** (solo
  spec/analysis). Hueco de evidencia **real**, no de parseo: L4 lo pone
  en la mesa con denominador en vez de esconderlo.

## ③ L5 — registro numérico ↔ paper ↔ addendum

In [ ]:
import sys, json
sys.path.insert(0, ".")
from numeric_join import run as num_run
r = num_run()
print("== join_summary (paper_data numbers= ↔ main.tex ↔ addendum.tex) ==")
print(json.dumps(r["join_summary"], indent=1))
print()
print("== tokens en NINGUNO de los dos documentos ==")
for row in r["join_rows"]:
    if row["value"] and not row["in_paper"] and not row["in_addendum"]:
        print(f"  exp{row['id']}  {row['value']:10s}  "
              f"label={row.get('label') or '-'}")
print()
print("== cruce directo addendum ↔ paper (3 anclas) ==")
for c in r["cross"]:
    print(f"  [{c['verdict']:14s}] {c['id']:22s} {c['note'][:90]}")

== join_summary (paper_data numbers= ↔ main.tex ↔ addendum.tex) ==
{
 "entries_total": 17,
 "entries_with_numbers": 13,
 "entries_without_numbers": [
  "4",
  "5",
  "6",
  "11"
 ],
 "tokens_checked": 32,
 "in_paper": 30,
 "in_addendum": 5,
 "in_both": 5,
 "in_neither": 2,
 "ratio_paper": 0.9375,
 "ratio_addendum": 0.1562,
 "ratio_both": 0.1562
}

== tokens en NINGUNO de los dos documentos ==
  exp10  1.3499      label=7962→
  exp10  0.1214      label=ΔH=

== cruce directo addendum ↔ paper (3 anclas) ==
  [match         ] ns_ticks_N             N=3000 ticks por nivel de ruido (EXP-NS-01)
  [match_phrase  ] sigma_noise_bound      addendum escribe 4.0‖x‖ (convención noise=4×norm); el paper lo dice en frase 'four times t
  [match         ] sigma_in_convention    σ_in ∈ [0,2.0] (param del sweep) — coexiste con noise=4×norm; NO es contradicción si se le


In [ ]:
# layer2a: abstract ↔ body (mismos candidatos, join por oración exacta)
r2 = num_run()
print(json.dumps(r2["layer2a"], indent=1))

{
 "abstract_candidates": 10,
 "numeric_total": 8,
 "numeric_checked": 4,
 "numeric_matched": 4,
 "numeric_miss": [],
 "theorem_total": 2,
 "theorem_matched": 0,
 "theorem_miss": [
  "(6)~under \\emph{sustained} bounded noise the kernel never leaves the f",
  "(8)~with all five Beer systems running live, each engages within its o"
 ],
 "ratio_numeric": 1.0,
 "ratio_theorem": 0.0
}


### Lectura — L5

- **17 entradas** en el registry, **13 con `numbers=`** (las 4 sin números
  — exp 4/5/6/11 — son el gap honesto del registro); **32 tokens**:
  **en paper 30 (0.9375)**, en addendum 5 (0.1562), en ambos 5.
- **En ninguno: 2 — exp10 (Maxwell)**: el registry guarda los números
  del **toy** (`1.3499`, `0.1214`) y el paper cita los del **kernel real**
  (`1.4865`, `0.3097`, main.tex L2379/L2417). El propio registry ya
  registra los reales en otro campo (L261) — `numbers=` está desfasado.
  Drift honesto, no error de parseo (la variante `4.00→4.0` sí matchea).
- **Cruce 3/3:** `N=3000` literal en ambos; el límite de ruido el addendum
  lo escribe `4.0‖x‖` y el paper «four times the state norm» —
  **misma cantidad, dos convenciones** (`σ_in∈[0,2.0]` es el parámetro
  del sweep, `noise=4×norm` la etiqueta); `σ_in=2.0` en ambos.
  **No es contradicción** si se leen las dos convenciones.
- **Layer2a:** abstract numérico **4/4 exactos** (ratio 1.0);
  teoremas **0/2** por oración *exacta* — techo honesto del método
  (paraphrase no matchea por diseño; no se finge cobertura).

In [ ]:
import subprocess, sys
chk = subprocess.run([sys.executable, "-m", "pytest", "--version"],
                     capture_output=True, text=True)
if chk.returncode != 0:
    print("pytest no disponible en este env — suite 30/30 en local (R1 locality)")
else:
    r = subprocess.run(
        [sys.executable, "-m", "pytest",
         "test_l3_verify.py", "test_hypo_join.py", "test_numeric_join.py",
         "-q"],
        capture_output=True, text=True)
    print((r.stdout or r.stderr)[-900:])
    print("rc =", r.returncode, "(0 = 30/30 PASS)")

..............................                                           [100%]
=============================== warnings summary ===============================
docs/papers/attractor_convergence/analysis/test_l3_verify.py: 1694 warnings
  /usr/lib/python3/dist-packages/mpmath/libmp/libintmath.py:75: DeprecationWarning: bitcount function is deprecated
    warnings.warn("bitcount function is deprecated",

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
30 passed, 1694 warnings in 12.22s

rc = 0 (0 = 30/30 PASS)


## Estado y siguiente

- **NB07 listo:** L3 cerrado (9 checks: 7 PASS + 1 CONDITIONAL +
  1 PASS_GRID), L4 con denominador honesto (11/15 = 0.7333, gap
  EXP-TC-01 sin results.vsm), L5 registry-driven (0.9375 en paper +
  cruce 3/3 + layer2a 4/4 numérico) — **L1→L5 del plan auto-ordenado
  COMPLETA** (NB02→NB07), 30/30 tests de esta capa.
- **Hallazgos para rmw3/S5:** (a) exp10 `numbers=` = toy, paper = kernel
  real — actualizar registry; (b) EXP-TC-01 sin `results.vsm` — hueco de
  evidencia térmica; (c) convención σ_in 2.0 vs 4.0‖x‖ documentada, no
  contradicción. Nada de esto se envía a Fierro sin orden S5.
- **Siguiente (queue_exec_plan):** F4 — verificar y flippear las 3
  tareas; F5 — `scripts/doc_tracking/` (verify_docs.py +
  doc_changelog.jsonl). Sigue pendiente S5: graph proposal (D3/D4),
  freeze v0.9 + OMEGA-MARGIN, envío a Fierro.